# 02 - Limpeza e Qualidade de Dados (Silver)

## Contexto

Este notebook implementa a camada **Silver**: parte da tabela Bronze (dado bruto, tudo como `string`) e produz uma tabela única, limpa e tipada: `silver.steam_games.games`.

Antes da limpeza, o notebook faz uma varredura sistemática de qualidade — completude, consistência, acurácia e outliers — em cada atributo do conjunto de dados que entra no projeto, e as decisões de limpeza tomadas a seguir são resultado direto dessas checagens.

Colunas fora do escopo das perguntas de negócio do projeto (descrições longas, URLs, imagens, vídeos, idiomas, conquistas, tempo de jogo, `score_rank`/`user_score` quase inteiramente vazios, `tags`, `developers`/`publishers`) não entram nessa varredura — foram descartadas já na definição do objetivo (Etapa 1), então não fazem parte do "conjunto de dados do projeto" a partir daqui.

In [0]:
# Importa as funções do Spark e carrega a tabela Bronze que será analisada e limpa.
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

bronze_games_full = spark.table("bronze.steam_games.games_full_raw")

## Qualidade de Dados

### Perfil sistemático por atributo

Para cada atributo usado no projeto: completude (nulo/vazio), consistência (o valor tem o formato/tipo esperado) e acurácia (o valor, já no tipo certo, faz sentido dentro do domínio esperado). Unicidade e outliers têm lógica própria e vêm nas seções seguintes.

In [0]:
# Perfil de qualidade por atributo: para cada coluna usada no projeto, conta quantos valores são nulos/vazios
# (completude), quantos não têm o formato ou tipo esperado (consistência) e quantos estão fora do domínio válido
# (acurácia). O resultado de todas as colunas vira uma única tabela.
# (coluna, tipo esperado, domínio mínimo, domínio máximo, categorias válidas — quando aplicável)
ATTR_CONFIG = [
    ("appid",             "int",    1,    None, None),
    ("name",              "string", None, None, None),
    ("release_date",      "date",   None, None, None),
    ("required_age",      "int",    0,    100,  None),
    ("price",             "double", 0,    1000, None),
    ("discount",          "double", 0,    100,  None),
    ("dlc_count",         "int",    0,    None, None),
    ("windows",           "bool",   None, None, ["true", "false"]),
    ("mac",               "bool",   None, None, ["true", "false"]),
    ("linux",             "bool",   None, None, ["true", "false"]),
    ("metacritic_score",  "int",    0,    100,  None),
    ("genres",            "list",   None, None, None),
    ("categories",        "list",   None, None, None),
    ("positive",          "int",    0,    None, None),
    ("negative",          "int",    0,    None, None),
    # -1 é uma sentinela de "sem dado" usada pela própria fonte nestes dois campos — ver achado abaixo.
    ("pct_pos_total",     "double", -1,   100,  None),
    ("num_reviews_total", "int",    -1,   None, None),
    ("estimated_owners",  "range",  None, None, None),
    ("peak_ccu",          "int",    0,    None, None),
]


def profile_column(df, col_name, tipo, dmin, dmax, categorias):
    c = F.col(col_name)
    is_blank = c.isNull() | (F.trim(c) == "")

    casted = None
    if tipo == "int":
        casted = c.cast("int")
    elif tipo == "double":
        casted = c.cast("double")
    elif tipo == "date":
        casted = F.to_date(c, "yyyy-MM-dd")

    if tipo in ("int", "double", "date"):
        inconsistente = (~is_blank) & casted.isNull()
    elif tipo == "bool":
        inconsistente = (~is_blank) & (~F.lower(F.trim(c)).isin(categorias))
    elif tipo == "list":
        inconsistente = (~is_blank) & ~(F.trim(c).startswith("[") & F.trim(c).endswith("]"))
    elif tipo == "range":
        inconsistente = (~is_blank) & (~c.rlike(r"^\d+\s*-\s*\d+$"))
    else:
        inconsistente = F.lit(False)

    fora_dominio = F.lit(False)
    if tipo in ("int", "double"):
        if dmin is not None:
            fora_dominio = fora_dominio | (casted.isNotNull() & (casted < dmin))
        if dmax is not None:
            fora_dominio = fora_dominio | (casted.isNotNull() & (casted > dmax))

    row = df.select(
        F.count("*").alias("total"),
        F.sum(F.when(is_blank, 1).otherwise(0)).alias("nulos"),
        F.sum(F.when(inconsistente, 1).otherwise(0)).alias("inconsistentes"),
        F.sum(F.when(fora_dominio, 1).otherwise(0)).alias("fora_dominio"),
    ).first()

    total = row["total"]
    return {
        "atributo": col_name,
        "tipo_esperado": tipo,
        "nulos": row["nulos"],
        "pct_nulos": round(row["nulos"] / total * 100, 1),
        "inconsistentes": row["inconsistentes"],
        "fora_dominio": row["fora_dominio"],
    }


perfil = [profile_column(bronze_games_full, *cfg) for cfg in ATTR_CONFIG]
display(spark.createDataFrame(perfil))

atributo,fora_dominio,inconsistentes,nulos,pct_nulos,tipo_esperado
appid,0,0,0,0.0,int
name,0,0,2,0.0,string
release_date,0,0,0,0.0,date
required_age,1,0,0,0.0,int
price,0,0,0,0.0,double
discount,0,0,0,0.0,double
dlc_count,0,0,0,0.0,int
windows,0,0,0,0.0,bool
mac,0,0,0,0.0,bool
linux,0,0,0,0.0,bool


### Outliers

O critério clássico de outlier (`Q3 + 1,5×IQR` ou `3×IQR`) quebra em várias colunas deste conjunto: `discount`, `dlc_count`, `peak_ccu` e `required_age` têm tantos zeros que o próprio 1º e 3º quartil caem em zero — qualquer valor positivo viraria "outlier", o que não ajuda em nada. Por isso o critério usado aqui é por **percentil**: valores acima do percentil 99,9% (os 0,1% mais extremos da coluna) — não depende dos quartis e não quebra com colunas concentradas em zero. `pct_pos_total`/`num_reviews_total` ficam de fora porque a sentinela `-1` (achado abaixo) invalidaria o cálculo; `estimated_owners` também, por já ser uma faixa categórica.

In [0]:
# Outliers: para cada coluna numérica, calcula os percentis 99 e 99,9 (ignorando valores sentinela) e conta
# quantos valores ficam acima do percentil 99,9, ou seja, os 0,1% mais extremos.
outlier_cols = ["price", "discount", "dlc_count", "required_age", "metacritic_score", "positive", "negative", "peak_ccu"]
# valores-sentinela que não são dado real e por isso são ignorados antes do cálculo do percentil
sentinelas = {"metacritic_score": 0, "required_age": -1}

outliers = []
for col_name in outlier_cols:
    casted = F.col(col_name).cast("double")
    base = bronze_games_full
    if col_name in sentinelas:
        base = base.filter(casted != sentinelas[col_name])
    valores = base.select(casted.alias("v")).na.drop()
    p99, p999 = valores.approxQuantile("v", [0.99, 0.999], 0.0) 
    maximo = valores.agg(F.max("v")).first()[0]
    n_outliers = valores.filter(F.col("v") > p999).count()
    outliers.append({
        "atributo": col_name, "p99": round(p99, 2), "p99_9": round(p999, 2),
        "maximo": maximo, "outliers_acima_p99_9": n_outliers,
    })

display(spark.createDataFrame(outliers))

atributo,maximo,outliers_acima_p99_9,p99,p99_9
price,999.98,8,39.99,199.99
discount,100.0,1,85.0,95.0
dlc_count,3427.0,89,7.0,49.0
required_age,21.0,2,13.0,18.0
metacritic_score,97.0,3,91.0,94.0
positive,7480813.0,94,15726.0,178316.0
negative,1135108.0,94,2619.0,24953.0
peak_ccu,1212356.0,94,411.0,12239.0


`positive`, `negative` e `peak_ccu` têm outliers pelo critério de percentil, e isso é esperado, não é problema de dado: o mercado da Steam é extremamente concentrado (poucos jogos, como Counter-Strike 2, respondem por milhões de reviews e picos de jogadores, enquanto a maioria tem poucos).

`discount`, `required_age` e `metacritic_score` têm poucos outliers (1, 2 e 3 respectivamente) e também são valores reais, não erro.

`dlc_count` e `price` têm achados mais específicos, vale olhar os casos.

In [0]:
# Lista os 5 jogos com maior dlc_count para inspecionar os outliers dessa coluna.
display(
    bronze_games_full
    .select("appid", "name", F.col("dlc_count").cast("int").alias("dlc_count"))
    .orderBy(F.desc("dlc_count"))
    .limit(5)
)

appid,name,dlc_count
1196310,Fantasy Grounds VTT,3427
252690,Fantasy Grounds Classic,2004
221680,Rocksmith® 2014 Edition REMASTERED LEARN & PLAY,1191
363890,RPG Maker MV,954
1096900,RPG Maker MZ,767


Os jogos com `dlc_count` mais extremo são plataformas de tabuleiro virtual (Fantasy Grounds) e um simulador com dezenas de pacotes de música (Rocksmith) — cada complemento pequeno é registrado como um "DLC" separado na Steam. Não é erro de coleta, mas são outliers reais que pesam na média de qualquer agregação por faixa de `dlc_count` (relevante para a pergunta 6 do objetivo).

In [0]:
# Lista os 5 jogos com maior preço para inspecionar os outliers dessa coluna.
display(
    bronze_games_full
    .select("appid", "name", F.col("price").cast("double").alias("price"))
    .orderBy(F.desc("price"))
    .limit(5)
)

appid,name,price
2499620,The Leverage Game,999.98
2504210,The Leverage Game Business Edition,999.98
1200520,Ascent Free-Roaming VR Experience,999.0
3013840,True Love,500.0
253670,Aartform Curvy 3D 3.0,299.9


Os preços mais extremos (até $999,98) também não são erro: são softwares de nicho com precificação empresarial vendidos pela mesma vitrine dos jogos (ex.: "The Leverage Game Business Edition", "Ascent Free-Roaming VR Experience" — um produto de licenciamento para operadores de arcade VR). Vale notar que "The Leverage Game" aparece duas vezes (edição normal e "Business Edition") com o mesmo preço — mais um caso do achado de fichas de loja repetidas sob `appid`s diferentes, visto na seção de Unicidade.

### Unicidade

`appid` é a chave do conjunto de dados — duplicidade nela seria um erro grave. Também vale checar duplicidade de **conteúdo** (mesma ficha de loja sob `appid`s diferentes) e a presença de builds de teste ("Playtest"), que não são o produto real.

In [0]:
# Unicidade: agrupa por appid e conta quantos aparecem mais de uma vez; também conta as entradas com "Playtest"
# no nome.
dup_ids = bronze_games_full.groupBy("appid").count().filter("count > 1").count()
playtest_count = bronze_games_full.filter(
    F.lower(F.coalesce(F.col("name"), F.lit(""))).contains("playtest")
).count()
print("appid duplicados:", dup_ids)
print("entradas com 'Playtest' no nome:", playtest_count)

appid duplicados: 0
entradas com 'Playtest' no nome: 5219


In [0]:
# Agrupa os jogos por todos os campos descritivos da ficha de loja (tudo menos o appid) e conta os grupos com
# mais de uma linha, ou seja, o mesmo jogo cadastrado sob appids diferentes.
# Duplicidade de conteúdo: mesma ficha de loja (nome, descrição, preço, gêneros...)
# repetida sob appids diferentes — a Steam faz isso para variantes de edição/pacote/SKU.
content_cols = [
    "name", "release_date", "detailed_description", "about_the_game", "short_description",
    "developers", "publishers", "price", "dlc_count", "required_age", "genres", "categories",
]
conteudo_duplicado = (
    bronze_games_full
    .filter(F.trim(F.coalesce(F.col("detailed_description"), F.lit(""))) != "")
    .groupBy(content_cols)
    .count()
    .filter("count > 1")
)
print("grupos de fichas de loja idênticas sob appids diferentes:", conteudo_duplicado.count())
print("linhas envolvidas:", conteudo_duplicado.agg(F.sum("count")).collect()[0][0])

grupos de fichas de loja idênticas sob appids diferentes: 28
linhas envolvidas: 82


### Achado adicional: `positive`/`negative` (Steam API) e `pct_pos_total`/`num_reviews_total` (SteamSpy) têm cobertura diferente

No perfil sistemático, `pct_pos_total`/`num_reviews_total` foram configurados com domínio a partir de `-1` justamente porque esse valor é uma sentinela de "sem dado" usada pela própria fonte — por isso aparecem com 0 valores fora do domínio. Investigando o que essa sentinela esconde: essas duas fontes de review dentro do mesmo arquivo nem sempre concordam sobre quais jogos têm dado disponível. Usar só uma das duas perderia cobertura de milhares de jogos.

In [0]:
# Cruza as duas fontes de review do arquivo: para cada jogo verifica se positive+negative (Steam API) e
# pct_pos_total (SteamSpy) têm dado, e soma quantos jogos caem em cada combinação (só uma tem dado, nenhuma, ou
# as duas).
posneg = (F.col("positive").cast("double") + F.col("negative").cast("double"))
pct_ausente = (F.col("pct_pos_total").cast("double") == -1)

cobertura = bronze_games_full.select(
    F.sum(F.when((posneg == 0) & (~pct_ausente), 1).otherwise(0)).alias("so_pct_pos_total_tem_dado"),
    F.sum(F.when(pct_ausente & (posneg > 0), 1).otherwise(0)).alias("so_positive_negative_tem_dado"),
    F.sum(F.when((posneg == 0) & pct_ausente, 1).otherwise(0)).alias("nenhum_tem_dado"),
    F.sum(F.when((posneg > 0) & (~pct_ausente), 1).otherwise(0)).alias("os_dois_tem_dado"),
)
display(cobertura)

so_pct_pos_total_tem_dado,so_positive_negative_tem_dado,nenhum_tem_dado,os_dois_tem_dado
11248,28523,11052,44125


**Achados e decisões:**

| Achado | Quantidade | Decisão |
|---|---|---|
| `metacritic_score == 0` | 91.372 de 94.948 (96,2%) | Tratado como nulo — 0 é usado como sentinela de "sem nota", não é uma nota real (não aparece como "fora de domínio" no perfil porque 0 está dentro de 0-100; é um achado semântico, não de intervalo) |
| `required_age < 0` | 1 registro (`Ys X: Nordics`, appid 2731870, valor -1) | Tratado como nulo — idade negativa é inválida (aparece no perfil como "fora de domínio") |
| Nomes contendo "Playtest" | 5.219 de 94.948 (5,5%) | Removidos — são builds de teste, não o jogo real (o próprio dataset avisa sobre isso) |
| `name` nulo/vazio | 2 registros (confirmado no perfil sistemático) | Mantidos e rotulados como "(nome não informado)" — o resto dos dados (preço, gênero, reviews) é real e utilizável; filtro de "Playtest" usa `coalesce` para não descartá-los por engano |
| `appid` duplicado | 0 | Nenhuma ação necessária |
| `discount`, `price`, `positive`, `negative`, `peak_ccu`, `dlc_count`, `windows`/`mac`/`linux`, `genres`/`categories`, `estimated_owners` fora do domínio ou formato esperado (perfil sistemático) | 0 em todos | Nenhuma ação necessária — a fonte é consistente nesses atributos |
| Mesma ficha de loja (nome, descrição, preço, gêneros...) repetida sob `appid`s diferentes | 82 linhas em 28 grupos (0,09%) | Mantidas — a Steam cria múltiplos `appid` para variantes de edição/pacote/SKU do mesmo jogo (ex.: "Shadow of the Tomb Raider: Definitive Edition" aparece em 20 `appid`s); cada um é uma listagem real e independente, com preço e reviews próprios. Não é linha duplicada, é limitação conhecida: perguntas agregadas por gênero/preço/DLC dão peso maior a franquias com muitas variantes de SKU |
| Outliers de `dlc_count` acima do percentil 99,9% | 89 (~0,1%, por definição do critério); os 3 mais extremos: Fantasy Grounds VTT (3.427), Fantasy Grounds Classic (2.004), Rocksmith 2014 Edition REMASTERED (1.191) | Mantidos — são produtos legítimos que vendem centenas de pequenos complementos como "DLC" na Steam, não é erro de coleta; documentado porque pesa na média de popularidade por faixa de DLC na análise (pergunta 6) |
| Outliers de `price` acima do percentil 99,9% | 8 jogos, até $999,98 (The Leverage Game Business Edition, Ascent Free-Roaming VR Experience, entre outros) | Mantidos — são softwares de nicho/empresariais vendidos na mesma vitrine dos jogos, preço real, não erro de coleta |
| Outliers de `positive`/`negative`/`peak_ccu` acima do percentil 99,9% | 94 cada (mercado da Steam é extremamente concentrado) | Mantidos — reflete a real distribuição do mercado (poucos hits dominam), não erro de dado; ressalva para qualquer estatística de média simples nessas colunas |
| `positive`/`negative` chega a 0 mas `pct_pos_total` tem dado real | 11.248 (11,8%) | Métrica de aceitação (`acceptance_ratio`) cai para `pct_pos_total` nesses casos |
| `pct_pos_total` ausente (`-1`) mas `positive`/`negative` tem dado real | 28.523 (30,0%) | Métrica de aceitação usa `positive`/`negative` nesses casos (é o valor padrão, por ser mais granular) |
| Nenhuma das duas fontes de review tem dado | 11.052 (11,6%) | `acceptance_ratio` fica nulo — de fato não há avaliação disponível |

Resultado: `acceptance_ratio` cobre 88,4% dos jogos (83.896 de 94.948), contra 76,5% usando só `positive`/`negative` ou 58,3% usando só `pct_pos_total` isoladamente. A mesma lógica de fallback é aplicada em `review_count` (volume de reviews), usando `num_reviews_total` só quando `positive`+`negative` for zero.

`positive` e `negative` (contagens brutas da Steam API) não entram na tabela final: nenhuma pergunta de negócio usa contagem absoluta, só a proporção (`acceptance_ratio`) e o volume (`review_count`) — e manter as contagens brutas ao lado de uma métrica que às vezes vem de outra fonte criaria linhas visualmente contraditórias (ex.: `positive=0, negative=0, acceptance_ratio=66`, quando os 66% vieram do SteamSpy, não da Steam API). Em vez disso, a coluna `review_data_source` registra a linhagem: de qual das duas APIs `acceptance_ratio`/`review_count` foram calculados (`steam_api`, `steamspy` ou nulo quando nenhuma tem dado).

`genres` e `categories` chegam como texto no formato de lista Python (`"['Action', 'Adventure']"`) e são convertidas para `array<string>`. `estimated_owners` chega como uma faixa em texto (`"100000 - 200000"`) e é convertida em limite mínimo, máximo e médio.

In [0]:
# Constrói a tabela Silver a partir da Bronze: remove as entradas de Playtest, converte cada coluna para o tipo
# correto, transforma valores sentinela em nulo, converte as listas em texto (genres/categories) em arrays,
# separa a faixa estimated_owners em mínimo/máximo/média e calcula acceptance_ratio e review_count. Por fim
# grava em silver.steam_games.games.
# Funções auxiliares: parse_list converte texto no formato de lista Python ("['Action', 'Indie']") em array de strings;
# to_bool converte o texto "true"/"false" em booleano.
list_schema = ArrayType(StringType())


def parse_list(col):
    return F.from_json(F.regexp_replace(col, "'", '"'), list_schema)


def to_bool(col):
    return F.when(F.lower(F.trim(col)) == "true", True).when(F.lower(F.trim(col)) == "false", False)


# Métrica de aceitação: usa positive/negative (Steam API) quando somam mais de 0; caso contrário, cai para pct_pos_total
# (SteamSpy) se ele tiver dado (-1 significa "sem dado"). review_count segue a mesma lógica e review_data_source registra a origem.
positive_d = F.col("positive").cast("double")
negative_d = F.col("negative").cast("double")
pct_pos_total_d = F.col("pct_pos_total").cast("double")
num_reviews_total_d = F.col("num_reviews_total").cast("double")

tem_dado_api = (positive_d + negative_d) > 0
tem_dado_steamspy = pct_pos_total_d != -1

acceptance_ratio = F.when(
    tem_dado_api, positive_d * 100.0 / (positive_d + negative_d)
).when(tem_dado_steamspy, pct_pos_total_d)

review_count = F.when(tem_dado_api, positive_d + negative_d).when(
    num_reviews_total_d != -1, num_reviews_total_d
)

review_data_source = F.when(tem_dado_api, F.lit("steam_api")).when(tem_dado_steamspy, F.lit("steamspy"))

# Monta a tabela Silver: filtra Playtest, converte tipos e aplica as regras de limpeza coluna a coluna.
silver_games = (
    bronze_games_full
    .filter(~F.lower(F.coalesce(F.col("name"), F.lit(""))).contains("playtest"))
    .select(
        F.col("appid").cast("int").alias("app_id"),
        F.when(F.trim(F.coalesce(F.col("name"), F.lit(""))) == "", F.lit("(nome não informado)"))
        .otherwise(F.trim(F.col("name")))
        .alias("title"),
        F.to_date(F.col("release_date"), "yyyy-MM-dd").alias("release_date"),
        F.when(F.col("required_age").cast("int") >= 0, F.col("required_age").cast("int")).alias("required_age"),
        F.col("price").cast("double").alias("price"),
        F.col("discount").cast("double").alias("discount"),
        F.col("dlc_count").cast("int").alias("dlc_count"),
        to_bool(F.col("windows")).alias("windows"),
        to_bool(F.col("mac")).alias("mac"),
        to_bool(F.col("linux")).alias("linux"),
        F.when(F.col("metacritic_score").cast("int") > 0, F.col("metacritic_score").cast("int")).alias("metacritic_score"),
        parse_list(F.col("genres")).alias("genres"),
        parse_list(F.col("categories")).alias("categories"),
        acceptance_ratio.alias("acceptance_ratio"),
        review_count.alias("review_count"),
        review_data_source.alias("review_data_source"),
        F.split(F.col("estimated_owners"), " - ").getItem(0).cast("long").alias("estimated_owners_min"),
        F.split(F.col("estimated_owners"), " - ").getItem(1).cast("long").alias("estimated_owners_max"),
        F.col("peak_ccu").cast("int").alias("peak_ccu"),
    )
    .withColumn(
        "estimated_owners_avg",
        (F.col("estimated_owners_min") + F.col("estimated_owners_max")) / F.lit(2.0),
    )
    .dropDuplicates(["app_id"])
    .withColumn("_silver_processed_at", F.current_timestamp())
)

(
    silver_games.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.steam_games.games")
)

print("silver.steam_games.games:", spark.table("silver.steam_games.games").count(), "linhas")

silver.steam_games.games: 89729 linhas


## Validação

Contagem esperada (validada localmente antes da subida): **89.729** linhas (94.948 menos as 5.219 entradas de "Playtest", mantendo os 2 registros de nome vazio).

In [0]:
# Conta as linhas da tabela Silver (esperado: 89.729).
display(spark.sql("SELECT COUNT(*) AS linhas FROM silver.steam_games.games"))

linhas
89729
